# 01 · Corpus — pre-chunked CSVs to even shards on S3

**Runs on: wherever the corpus files are.** No GPU, no `kubectl`, no
Elasticsearch — the corpus is already chunked, so this notebook only validates,
re-shards and uploads.

Expects the layout:

```
<FS_CORPUS_ROOT>/
├── abstracts/clusters/abstracts_cluster_*.csv
├── guides/chunks_guide_*.csv
└── textbooks/chunks_textbook_*.csv
```

Two things about that layout drive what this notebook does.

`iter_chunks` treats a directory as a flat corpus — `glob("*.csv")`, not
recursive — so pointing the library at the root finds nothing. The three source
directories are walked explicitly.

The natural files are wildly uneven: a textbook CSV is thousands of chunks and
a one-page guide leaflet is a few dozen. Sharding on file boundaries would hand
one worker a textbook and another a poster, and the GPU box would sit idle
waiting for the long pole. So the corpus is **re-sharded evenly by row count**
across all three sources.

## 0. Preflight

```bash
export FS_CORPUS_ROOT=/path/to/corpus      # the directory in the tree above
export S3_ACCESS_KEY=root
export S3_SECRET_KEY=...                   # minio_root secret, key `password`
export FS_RUN_ID=2026-09-21-full           # the SAME string on every machine
```

In [ ]:
import os, csv, json, time
from pathlib import Path

REQUIRED = ["FS_CORPUS_ROOT", "S3_ACCESS_KEY", "S3_SECRET_KEY", "FS_RUN_ID"]
missing = [v for v in REQUIRED if not os.environ.get(v)]
if missing:
    raise SystemExit(f"missing environment: {', '.join(missing)}")

ROOT        = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA        = ROOT / "data"
CORPUS_ROOT = Path(os.environ["FS_CORPUS_ROOT"]).expanduser().resolve()
WORK        = DATA / "handoff" / os.environ["FS_RUN_ID"]
(WORK / "corpus").mkdir(parents=True, exist_ok=True)

if not CORPUS_ROOT.is_dir():
    raise SystemExit(f"not a directory: {CORPUS_ROOT}")
print(f"corpus {CORPUS_ROOT}")
print(f"work   {WORK}")

In [ ]:
import os, json, hashlib
from pathlib import Path

try:
    import boto3
    from botocore.client import Config as BotoConfig
except ImportError as e:
    raise SystemExit("pip install boto3  # S3 handoff between the GPU box and the store box") from e

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "https://s3.wisefood-project.eu")
S3_BUCKET   = os.environ.get("S3_BUCKET", "foodscholar-graph-build")
RUN_ID      = os.environ["FS_RUN_ID"]          # same string on both machines

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    # MinIO speaks path-style; virtual-host style would resolve
    # <bucket>.s3.wisefood-project.eu, which has no DNS record.
    config=BotoConfig(signature_version="s3v4", s3={"addressing_style": "path"}),
)

def s3_key(*parts: str) -> str:
    return "/".join(["runs", RUN_ID, *parts])

def s3_exists(key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except Exception:
        return False

def s3_put(local: Path, key: str) -> None:
    s3.upload_file(str(local), S3_BUCKET, key)

def s3_get(key: str, local: Path) -> Path:
    local.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(S3_BUCKET, key, str(local))
    return local

def s3_list(prefix: str) -> list[str]:
    keys, token = [], None
    while True:
        kw = {"Bucket": S3_BUCKET, "Prefix": prefix}
        if token:
            kw["ContinuationToken"] = token
        resp = s3.list_objects_v2(**kw)
        keys.extend(o["Key"] for o in resp.get("Contents", []))
        if not resp.get("IsTruncated"):
            return sorted(keys)
        token = resp["NextContinuationToken"]

print(f"s3  {S3_ENDPOINT}/{S3_BUCKET}")
print(f"run {RUN_ID}")

## 1. Discover and validate

Every CSV is checked against the corpus contract before anything is uploaded.
A file with the wrong header fails here, on the machine holding the corpus,
rather than on the GPU box an hour into a run.

In [ ]:
from foodscholar.corpus.chunker import CORPUS_COLUMNS
REQUIRED_COLUMNS = {"chunk_id", "chunk_text", "type", "chunk_metadata"}

SOURCES = {
    "abstract": CORPUS_ROOT / "abstracts" / "clusters",
    "guide":    CORPUS_ROOT / "guides",
    "textbook": CORPUS_ROOT / "textbooks",
}

files, problems = {}, []
for label, d in SOURCES.items():
    if not d.is_dir():
        problems.append(f"{label}: missing directory {d}")
        files[label] = []
        continue
    found = sorted(d.glob("*.csv"))
    files[label] = found
    if not found:
        problems.append(f"{label}: no CSVs under {d}")

print(f"corpus contract: {CORPUS_COLUMNS}\n")
for label, found in files.items():
    print(f"  {label:10s} {len(found):3d} files")

for label, found in files.items():
    for p in found:
        with p.open(newline="") as fh:
            header = set(next(csv.reader(fh), []))
        gap = REQUIRED_COLUMNS - header
        if gap:
            problems.append(f"{p.name}: missing columns {sorted(gap)}")

if problems:
    print("\n!! problems:")
    for x in problems:
        print("   ", x)
else:
    print("\nall files carry the required columns")

## 2. Count, and look for gaps

Numbered series are worth checking: a missing index is either an empty cluster
that was never written, or a file that went missing on the way here. The
difference matters and only you can tell them apart — this just says which
numbers are absent.

In [ ]:
import re

def row_count(p: Path) -> int:
    with p.open(newline="") as fh:
        return max(0, sum(1 for _ in fh) - 1)      # minus the header

inventory, totals = [], {}
t0 = time.perf_counter()
for label, found in files.items():
    n = 0
    for p in found:
        rows = row_count(p)
        n += rows
        inventory.append({"source": label, "path": str(p), "name": p.name, "rows": rows})
    totals[label] = n

for label, n in totals.items():
    print(f"  {label:10s} {n:7d} chunks in {len(files[label]):3d} files")
print(f"  {'TOTAL':10s} {sum(totals.values()):7d} chunks   ({time.perf_counter()-t0:.0f}s to count)")

# Gap check on the numbered abstract clusters.
nums = sorted(int(m.group(1))
              for p in files["abstract"]
              if (m := re.search(r"_(\d+)\.csv$", p.name)))
if nums:
    expected = set(range(min(nums), max(nums) + 1))
    gaps = sorted(expected - set(nums))
    print(f"\nabstract clusters {min(nums):02d}..{max(nums):02d}: {len(nums)} present")
    if gaps:
        print(f"  !! absent: {', '.join(f'{g:02d}' for g in gaps)}")
        print("     empty cluster, or a file that did not arrive? Check before running 02 —")
        print("     the graph will simply not contain whatever those abstracts covered.")

empty = [e for e in inventory if e["rows"] == 0]
if empty:
    print(f"\n  !! {len(empty)} file(s) with no rows: {', '.join(e['name'] for e in empty[:5])}")

## 3. Re-shard evenly

Shard size is a restart granularity, not a throughput knob — it is what the GPU
box redoes if a worker dies. 500 chunks is a couple of minutes of GPU work,
which is cheap to lose.

Rows are streamed file by file and cut at the shard boundary, so a large
textbook spans several shards and a small leaflet shares one. Source order is
preserved, so the sharding is deterministic and re-running produces identical
shards.

In [ ]:
SHARD_SIZE = int(os.environ.get("FS_SHARD_SIZE", "500"))
shard_dir = WORK / "corpus"

def write_shard(idx: int, rows: list[dict]) -> Path:
    p = shard_dir / f"shard_{idx:04d}.csv"
    if not p.exists() or p.stat().st_size == 0:
        with p.open("w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=CORPUS_COLUMNS, extrasaction="ignore")
            w.writeheader()
            w.writerows(rows)
    return p

shards, buf, idx, provenance = [], [], 0, {}
for entry in inventory:
    if entry["rows"] == 0:
        continue
    with Path(entry["path"]).open(newline="") as fh:
        for row in csv.DictReader(fh):
            buf.append(row)
            provenance.setdefault(idx, set()).add(entry["name"])
            if len(buf) >= SHARD_SIZE:
                shards.append({"shard": write_shard(idx, buf).name, "rows": len(buf),
                               "from": sorted(provenance[idx])})
                buf, idx = [], idx + 1
if buf:
    shards.append({"shard": write_shard(idx, buf).name, "rows": len(buf),
                   "from": sorted(provenance[idx])})

print(f"{len(shards)} shards of <= {SHARD_SIZE} chunks")
print(f"  total sharded: {sum(s['rows'] for s in shards)}")
for s in shards[:3]:
    print(f"    {s['shard']}  {s['rows']:4d}  from {', '.join(s['from'][:2])}"
          f"{' …' if len(s['from']) > 2 else ''}")

In [ ]:
# The shards must load as Chunks, or the GPU box discovers it after the upload.
from foodscholar.corpus import load_chunks

probe = load_chunks(shard_dir / shards[0]["shard"])
print(f"first shard loads: {len(probe)} chunks")
print(f"  id={probe[0].chunk_id}")
print(f"  source_type={probe[0].source_type}  doc={probe[0].source_doc_id[:60]}")
assert len(probe) == shards[0]["rows"], "shard row count does not match what loads"

## 4. Upload shards

In [ ]:
uploaded = 0
for s in shards:
    key = s3_key("corpus", s["shard"])
    s["key"] = key
    if s3_exists(key):
        continue
    s3_put(shard_dir / s["shard"], key)
    uploaded += 1
print(f"uploaded {uploaded}; {len(s3_list(s3_key('corpus')))} shards on s3")

## 5. Ship the ontology

40 MB of OWL and its parsed cache, so the GPU box provisions itself from S3
and needs nothing hand-copied.

In [ ]:
for f in ("foodon.owl", "foodon_cache.parquet", "foodon_cache.parquet.meta.json"):
    src = DATA / f
    if not src.exists():
        print(f"  !! missing {src}")
        continue
    key = s3_key("ontology", f)
    if s3_exists(key):
        print(f"  already on s3: {f}")
    else:
        t0 = time.perf_counter()
        s3_put(src, key)
        print(f"  uploaded {f} ({src.stat().st_size/1e6:.1f} MB, {time.perf_counter()-t0:.0f}s)")

In [ ]:
manifest_path = WORK / "manifest.json"
manifest_path.write_text(json.dumps({
    "run_id": RUN_ID,
    "corpus_root": str(CORPUS_ROOT),
    "shard_size": SHARD_SIZE,
    "chunks": sum(s["rows"] for s in shards),
    "by_source": totals,
    "source_files": {k: len(v) for k, v in files.items()},
    "shards": shards,
}, indent=2))
s3_put(manifest_path, s3_key("manifest.json"))

print(f"manifest: {sum(s['rows'] for s in shards)} chunks in {len(shards)} shards")
for k, v in totals.items():
    print(f"  {k:10s} {v:7d}")
print(f"\nNext: run 02_annotate on the GPU box with FS_RUN_ID={RUN_ID}")